In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/113-1-ntut-dl-app-midterm/IMDB_ntut_test.csv
/kaggle/input/113-1-ntut-dl-app-midterm/IMDB_ntut_train.csv
/kaggle/input/113-1-ntut-dl-app-midterm/IMDB_ntut_submit_example.csv


In [2]:
# 首先安裝需要的套件
!pip install torch transformers tqdm pandas numpy scikit-learn


[notice] A new release of pip is available: 23.0.1 -> 24.3.1
[notice] To update, run: pip install --upgrade pip


In [3]:
# Cell 1: 導入必要的函式庫
import pandas as pd
import torch
from transformers import BertTokenizer, BertModel
from torch.utils.data import Dataset, DataLoader
import numpy as np
from tqdm import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder


/usr/local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/usr/local/lib/python3.10/site-packages/torch_xla/__init__.py:202: UserWarning: `tensorflow` can conflict with `torch-xla`. Prefer `tensorflow-cpu` when using PyTorch/XLA. To silence this warning, `pip uninstall -y tensorflow && pip install tensorflow-cpu`. If you are in a notebook environment such as Colab or Kaggle, restart your notebook runtime afterwards.
  warnings.warn(


In [4]:
# Cell 2: 設置設備
# 檢查是否有可用的 GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"使用設備: {device}")


使用設備: cpu


In [5]:
# Cell 3: 數據載入
# 載入數據
print("讀取數據...")
try:
    train_df = pd.read_csv(r'/kaggle/input/113-1-ntut-dl-app-midterm/IMDB_ntut_train.csv', encoding='utf-8')
    test_df = pd.read_csv(r'/kaggle/input/113-1-ntut-dl-app-midterm/IMDB_ntut_test.csv', encoding='utf-8')
except UnicodeDecodeError:
    train_df = pd.read_csv(r'/kaggle/input/113-1-ntut-dl-app-midterm/IMDB_ntut_train.csv', encoding='latin1')
    test_df = pd.read_csv(r'/kaggle/input/113-1-ntut-dl-app-midterm/IMDB_ntut_test.csv', encoding='latin1')

讀取數據...


In [6]:
train_df

,review,sentiment
0,I borrowed (slightly modified) title from some...,positive
1,"I was the Production Accountant on this movie,...",positive
2,By far this has to be one of the worst movies ...,negative
3,Obviously inspired by Se7en and sometimes even...,positive
4,This movie is of almost generation-defining im...,positive
...,...,...
29995,`Shadow Magic' recaptures the joy and amazemen...,positive
29996,I found this movie to be quite enjoyable and f...,positive
29997,Avoid this one! It is a terrible movie. So wha...,negative
29998,This production was quite a surprise for me. I...,positive


In [7]:
test_df

,review
0,I really liked this Summerslam due to the look...
1,Not many television shows appeal to quite as m...
2,The film quickly gets to a major chase scene w...
3,Jane Austen would definitely approve of this o...
4,Expectations were somewhat high for me when I ...
...,...
19995,A touching movie about a talented woman who st...
19996,I just came from seeing this movie and decided...
19997,Dolemite is one of the best movies featuring a...
19998,"In the future, a disparate group of people asl..."


In [8]:
# 顯示數據基本信息
print("\n訓練集基本信息：")
print(train_df.info())
print("\n測試集基本信息：")
print(test_df.info())


訓練集基本信息：
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     30000 non-null  object
 1   sentiment  30000 non-null  object
dtypes: object(2)
memory usage: 468.9+ KB
None

測試集基本信息：
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   review  20000 non-null  object
dtypes: object(1)
memory usage: 156.4+ KB
None


In [9]:
# Cell 4: BERT 模型和 tokenizer 載入
# 載入 BERT tokenizer 和模型
print("載入 BERT 模型和 tokenizer...")
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')
model = model.to(device)
model.eval()

載入 BERT 模型和 tokenizer...


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

In [10]:
# Cell 5: 定義數據集類別
class IMDBDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length=512):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze()
        }

In [11]:
# Cell 6: 定義 BERT 嵌入函數
def get_bert_embeddings(texts, batch_size=8):
    dataset = IMDBDataset(texts, tokenizer)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    all_embeddings = []
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="處理批次"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )
            
            embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            all_embeddings.append(embeddings)
    
    return np.vstack(all_embeddings)


In [12]:
# Cell 7: 獲取 BERT 嵌入
print("處理訓練集...")
train_embeddings = get_bert_embeddings(train_df['review'])
print("處理測試集...")
test_embeddings = get_bert_embeddings(test_df['review'])

處理訓練集...


處理批次:  71%|███████▏  | 2681/3750 [1:17:24<30:51,  1.73s/it]


KeyboardInterrupt: 

In [ ]:
# 保存嵌入向量（可選）
print("保存嵌入向量...")
np.save('train_bert_embeddings.npy', train_embeddings)
np.save('test_bert_embeddings.npy', test_embeddings)

In [ ]:
# Cell 8: 相似度計算函數（可選的分析工具）
def cosine_similarity(v1, v2):
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

In [ ]:
# 計算示例相似度
print("\n示範評論相似度計算：")
n_examples = min(3, len(train_embeddings))
for i in range(n_examples):
    for j in range(i+1, n_examples):
        similarity = cosine_similarity(train_embeddings[i], train_embeddings[j])
        print(f"評論 {i+1} 和評論 {j+1} 的相似度: {similarity:.4f}")

In [ ]:
# Cell 9: 訓練分類器和進行預測
# 標籤編碼
le = LabelEncoder()
train_labels = le.fit_transform(train_df['sentiment'])

In [ ]:
# 訓練分類器
print("訓練分類器...")
classifier = LogisticRegression(max_iter=1000)
classifier.fit(train_embeddings, train_labels)

In [ ]:
# 進行預測
print("進行預測...")
predictions = classifier.predict(test_embeddings)
predictions_labels = le.inverse_transform(predictions)

In [ ]:
# Cell 10: 創建和保存提交文件
# 創建提交文件
print("創建提交文件...")
submit_df = pd.DataFrame({
    'Id': range(len(predictions_labels)),
    'Prediction': predictions_labels
})

In [ ]:
# 保存預測結果
submit_df.to_csv('IMDB_ntut_submit.csv', index=False)
print("預測結果已保存到 'IMDB_ntut_submit.csv'")

In [ ]:
# 顯示預測結果統計
print("\n預測結果統計：")
print(submit_df['Prediction'].value_counts())

In [ ]:
# 顯示前幾個預測結果
print("\n前5個預測結果：")
print(submit_df.head())

In [ ]:
# Cell 11: 額外的分析（可選）
print("\n嵌入向量的形狀:")
print(f"訓練集: {train_embeddings.shape}")
print(f"測試集: {test_embeddings.shape}")

In [ ]:
# 顯示分類器的性能指標
if 'sentiment' in train_df.columns:
    from sklearn.metrics import classification_report
    train_predictions = classifier.predict(train_embeddings)
    print("\n訓練集上的分類報告：")
    print(classification_report(train_labels, train_predictions))